Build user-item matrix.  Rows = users, columnts = movies. Values = ratings
This matrix is expect to be very sparse
Decompose Matrix into Latent Factors

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SVD-Recommender").getOrCreate()  
# create Spark session

ratings = spark.read.csv(
    "/home/rajesh/CSL7100/Assignment3/ml-latest-small/ratings.csv",
    header=True,
    inferSchema=True
)  
# load MovieLens ratings dataset

ratings = ratings.select("userId", "movieId", "rating")  
# keep only required columns for ALS

ratings.printSchema()  
# verify schema (userId:int, movieId:int, rating:double)

ratings.show(5)  
# preview dataset

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *



In [ ]:
user_item_matrix = ratings.groupBy("userId") \
    .pivot("movieId") \
    .agg(avg("rating"))
# convert long format ratings into user-item matrix using pivot

In [ ]:
#user_item_matrix.show(5)

In [ ]:
print((user_item_matrix.count(), len(user_item_matrix.columns)))

In [6]:
user_item_matrix = user_item_matrix.fillna(0)
# replace missing ratings with 0

In [7]:
#Normalize ratings
user_mean = ratings.groupBy("userId") \
    .agg(avg("rating").alias("user_mean"))
# calculate average rating per user

In [8]:
user_mean.show(5)

+------+------------------+
|userId|         user_mean|
+------+------------------+
|   148|3.7395833333333335|
|   463| 3.787878787878788|
|   471|             3.875|
|   496| 3.413793103448276|
|   243| 4.138888888888889|
+------+------------------+
only showing top 5 rows



In [9]:
print((user_mean.count(), len(user_mean.columns)))

(610, 2)


In [10]:
#Join with matrix.
user_item_matrix = user_item_matrix.join(user_mean, on="userId")
# attach user mean to normalize ratings

In [ ]:
#user_item_matrix.show(5)

In [ ]:
print((user_item_matrix.count(), len(user_item_matrix.columns)))

## 2 Appy SVD uisng numpy

In [ ]:
import numpy as np

matrix = user_item_matrix.drop("userId").toPandas().values
# convert Spark dataframe to NumPy matrix for SVD computation

In [ ]:
matrix.shape

In [ ]:
U, sigma, Vt = np.linalg.svd(matrix, full_matrices=False)
# perform singular value decomposition

In [ ]:
sigma_matrix = np.diag(sigma)
# convert singular values vector into diagonal matrix

In [ ]:
print("U shape:", U.shape)
print("Sigma shape:", sigma_matrix.shape)
print("Vt shape:", Vt.shape)

## 3 Predict missing ratings 

In [ ]:
k = 20  
# number of latent factors to keep (dimensionality reduction)

U_k = U[:, :k]  
# keep first k columns of U (top k user factors)

sigma_k = sigma_matrix[:k, :k]  
# keep top k singular values (importance of factors)

Vt_k = Vt[:k, :]  
# keep first k rows of Vᵀ (top k movie factors)

In [ ]:
approx_matrix = np.dot(np.dot(U_k, sigma_k), Vt_k)
# reconstruct approximate user-item rating matrix using reduced factors

In [ ]:
print(approx_matrix.shape)
# should match original matrix size (users × movies)

In [ ]:
import pandas as pd

approx_df = pd.DataFrame(
    approx_matrix,
    columns=user_item_matrix.drop("userId").columns
)
# convert reconstructed matrix to pandas dataframe with movieIds as columns

In [ ]:
approx_df.insert(0, "userId", user_item_matrix.select("userId").toPandas())
# add userId column back to dataframe

In [ ]:
user_id = 5  
# target user for recommendations

In [ ]:
user_predicted = approx_df[approx_df["userId"] == user_id].drop("userId", axis=1)
# extract predicted ratings for selected user

In [ ]:
recommended_movies = user_predicted.T.sort_values(
    by=user_predicted.index[0],
    ascending=False
)
# sort movies by predicted rating

In [ ]:
N = 10  
# number of recommendations

top_n_movies = recommended_movies.head(N)
# select top N movies with highest predicted rating

In [ ]:
print("Top Recommendations for User", user_id)
print(top_n_movies)